#03 - Modelado y selección de modelo

En este notebook se entrena y se compara dos algoritmos de clasificación para predecir el nivel de desempeño a partir de variables socioeconomicas y se selecciona el mejor teniendo presente la métrica F1-Macro evaluado.

In [1]:
#importamos la librerias y se carga el dataset #previamente transformado

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report

df = pd.read_csv("../data/processed/dataset_limpio.csv")
print(df.shape)

(1109, 11)


In [2]:
#Definición de variables de entrada X y objetivo Y
#X: variables socioeconómicas, 5 variables.
#Y: variable objetivo

features = ["estu_genero", "fami_estratovivienda", "fami_tieneinternet",
            "fami_educacionmadre", "fami_educacionpadre"]
X = df[features]
y = df["desempeno_global"]
print(X.shape, y.shape)

(1109, 5) (1109,)


In [3]:
# Separción de entranemiento y prueba
#80% para entrenar y 20% para probar random_state=42
#stratify=y manteiene la proporción.

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print("Train:", X_train.shape, "| Test:", X_test.shape)

Train: (887, 5) | Test: (222, 5)


In [4]:
#Modelo 1 - Regresión Lógica...

#Construimos un Pipeline con dos pasos: primero el OneHotEncoder (convierte las categorías
#en números, necesario para el modelo) y luego la Regresión Logística. El Pipeline se
#ajusta solo con los datos de entrenamiento, lo que evita la fuga de datos. Luego
#predecimos sobre el conjunto de prueba y reportamos F1-macro, accuracy y el reporte por
#clase.

pipe_lr = Pipeline([
    ("prep", OneHotEncoder(handle_unknown="ignore")),
    ("modelo", LogisticRegression(max_iter=1000, random_state=42))
])
pipe_lr.fit(X_train, y_train)
pred_lr = pipe_lr.predict(X_test)

print("=== Regresión Logística ===")
print("F1-macro:", round(f1_score(y_test, pred_lr, average="macro"), 3))
print("Accuracy:", round(accuracy_score(y_test, pred_lr), 3))
print(classification_report(y_test, pred_lr))

=== Regresión Logística ===
F1-macro: 0.582
Accuracy: 0.599
              precision    recall  f1-score   support

        Alto       0.67      0.83      0.74        87
        Bajo       0.61      0.52      0.56        52
       Medio       0.49      0.41      0.44        83

    accuracy                           0.60       222
   macro avg       0.59      0.59      0.58       222
weighted avg       0.59      0.60      0.59       222



In [5]:
#Modelo 2. Random Forest...

#Mismo procedimiento, pero con un Random Forest: un modelo más potente que combina muchos
#árboles de decisión. Lo evaluamos sobre el mismo conjunto de prueba para compararlo de
#forma justa con la Regresión Logística.

pipe_rf = Pipeline([
    ("prep", OneHotEncoder(handle_unknown="ignore")),
    ("modelo", RandomForestClassifier(random_state=42))
])
pipe_rf.fit(X_train, y_train)
pred_rf = pipe_rf.predict(X_test)

print("=== Random Forest ===")
print("F1-macro:", round(f1_score(y_test, pred_rf, average="macro"), 3))
print("Accuracy:", round(accuracy_score(y_test, pred_rf), 3))
print(classification_report(y_test, pred_rf))

=== Random Forest ===
F1-macro: 0.57
Accuracy: 0.581
              precision    recall  f1-score   support

        Alto       0.66      0.78      0.72        87
        Bajo       0.61      0.54      0.57        52
       Medio       0.45      0.40      0.42        83

    accuracy                           0.58       222
   macro avg       0.57      0.57      0.57       222
weighted avg       0.57      0.58      0.57       222



In [7]:
# Comparación de los modelos.
#Comparamos el F1-macro de ambos modelos sobre el conjunto de prueba y elegimos el que
#obtenga mejor desempeño. Ese será el modelo que serializaremos para el dashboard.

f1_lr = f1_score(y_test, pred_lr, average="macro")
f1_rf = f1_score(y_test, pred_rf, average="macro")
print(f"Regresión Logística F1-macro: {f1_lr:.3f}")
print(f"Random Forest F1-macro:       {f1_rf:.3f}")
print("Mejor modelo:", "Random Forest" if f1_rf >= f1_lr else "Regresión Logística")

Regresión Logística F1-macro: 0.582
Random Forest F1-macro:       0.570
Mejor modelo: Regresión Logística
